In [2]:

import os
from typing import TypedDict, Optional

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage  # adjust import path if needed
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

# 1. LLM setup
def get_groq_llm() -> ChatOpenAI:
    return ChatOpenAI(
        model="openai/gpt-oss-20b",
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0.7,
        max_tokens=2000
    )

llm = get_groq_llm()


In [3]:
# 2. Define State schema
class PostState(TypedDict, total=False):
    info: dict                     # input information for the post (topic, tone, audience, key points, etc)
    draft: str                     # current draft of LinkedIn post
    human_feedback: Optional[str]  # feedback from human reviewer
    approved: bool               # whether human reviewer approved
    final_post: Optional[str]      # final approved post content

In [ ]:
# 3. Node: Generate initial draft
def generate_draft(state: PostState) -> dict:
    info = state["info"]
    # Build prompt based on info
    prompt = f"""You are writing a LinkedIn post. Here is the info:
Topic: {info.get('topic')}
Key points: {info.get('key_points')}
Tone: {info.get('tone')}
Audience: {info.get('audience')}

Write a LinkedIn-style post (1-2 short paragraphs) with that info."""
    response = llm.invoke([HumanMessage(content=prompt)])
    draft = response.content.strip()
    print("=== AI Generated Draft ===")
    print(draft)
    return {"draft": draft, "approved": False}

# 4. Node: Ask human reviewer for feedback/approval
def ask_for_feedback(state: PostState) -> dict:
    print("\n--- PAUSING FOR HUMAN REVIEW ---")
    print("Draft to review:")
    print(state["draft"])
    # Use interrupt to pause and wait for human input
    feedback = interrupt("Please review the draft. Provide feedback (or type 'approved' if OK):")
    # Determine approval based on exact word (you may refine logic)
    approved = (feedback.strip().lower() == "approved")
    return {"human_feedback": feedback, "approved": approved}

# 5. Node: Decide next step
# def decide_next(state: PostState) -> Command:
#     if state.get("approved"):
#         return Command(goto="post_to_linkedin")
#     else:
#         return Command(goto="revise_draft")


def decide_next(state: PostState) -> str:
    if state.get("approved", False):
        return "approved"
    else:
        return "revise"


# 6. Node: Revise draft based on feedback
def revise_draft(state: PostState) -> dict:
    feedback = state["human_feedback"] or ""
    old_draft = state["draft"]
    prompt = f"""You are rewriting a LinkedIn post. Original draft:
{old_draft}

Feedback from reviewer:
{feedback}

Revise the draft accordingly (keeping same tone and audience) and output the improved post."""
    response = llm.invoke([HumanMessage(content=prompt)])
    new_draft = response.content.strip()
    print("=== AI Revised Draft ===")
    print(new_draft)
    return {"draft": new_draft, "approved": False}

# 7. Node: Post to LinkedIn (stub)
def post_to_linkedin(state: PostState) -> dict:
    final = state["draft"]
    # Here you would make the real LinkedIn API call.
    # e.g. linkedin_client.post_update(final)
    print("=== POSTING TO LINKEDIN ===")
    print(final)
    # After posting, you can set final_post
    return {"final_post": final}